# Practical Statistics for Data Scientists (Python)
# Chapter 6 — Statistical Machine Learning

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 6 explores key **statistical machine learning** methods that bridge classical statistics and modern ML.

| Topic | Key Concepts |
|-------|-------------|
| **K-Nearest Neighbors (KNN)** | Non-parametric; predict from nearby points |
| **Cross-Validation** | Honest model evaluation; k-fold, LOOCV |
| **Bias-Variance Trade-off** | Complexity vs generalisation |
| **Regularised Regression** | Ridge, Lasso, Elastic Net |
| **Model Interpretability** | SHAP values, partial dependence plots |

---


## 🔧 Setup & Imports

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import (cross_val_score, KFold,
                                      GridSearchCV, train_test_split)
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.pipeline import Pipeline

DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
LOAN3000  = DATA_BASE + "loan3000.csv"
LOAN_DATA = DATA_BASE + "loan_data.csv.gz"
HOUSE_CSV = DATA_BASE + "house_sales.csv"

np.random.seed(42)
print("Libraries loaded!")


---

## 1. K-Nearest Neighbors (KNN)

### 📚 Theory

**KNN** is a non-parametric, instance-based method:  
- **Classification:** Predict the majority class among the K nearest neighbors
- **Regression:** Predict the average target among the K nearest neighbors

**Distance metric (Euclidean):**
$$d(x, x') = \sqrt{\sum_{j=1}^{p} (x_j - x'_j)^2}$$

**Choosing K:**
- Small K → complex boundary → high variance, low bias
- Large K → smooth boundary → low variance, high bias
- Optimal K found by **cross-validation**

**Normalisation:** KNN is sensitive to feature scale — *always standardise features* before applying KNN.

> **Key Insight:** KNN has no training phase (lazy learning). Prediction is slow for large datasets. It captures non-linear boundaries but suffers from the "curse of dimensionality" as p increases.


In [ ]:
# KNN Classification on loan3000
loan3000 = pd.read_csv(LOAN3000)
loan3000['outcome'] = loan3000['outcome'].astype('category')

predictors = ['borrower_score', 'payment_inc_ratio']
X = loan3000[predictors]
y = (loan3000['outcome'] == 'default').astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Test different values of K
k_values = range(1, 51)
accuracies = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_scaled, y, cv=5, scoring='accuracy')
    accuracies.append(scores.mean())

plt.figure(figsize=(8, 4))
plt.plot(k_values, accuracies, 'b-o', markersize=4)
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('5-Fold CV Accuracy')
plt.title('KNN: Choosing Optimal K')
plt.axvline(np.argmax(accuracies) + 1, color='red', linestyle='--',
            label=f'Best K = {np.argmax(accuracies) + 1}')
plt.legend()
plt.tight_layout()
plt.show()

best_k = np.argmax(accuracies) + 1
print(f"Best K: {best_k}, CV Accuracy: {max(accuracies):.4f}")


In [ ]:
# KNN Decision Boundary
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_scaled, y)

xx, yy = np.meshgrid(
    np.linspace(X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5, 200),
    np.linspace(X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5, 200)
)
Z = best_knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdBu',
                     edgecolors='none', s=15, alpha=0.5)
ax.set_xlabel('Borrower Score (scaled)')
ax.set_ylabel('Payment-to-Income Ratio (scaled)')
ax.set_title(f'KNN Decision Boundary (K={best_k})')
plt.tight_layout()
plt.show()


---

## 2. Cross-Validation

### 📚 Theory

**Cross-Validation (CV)** provides an honest estimate of model performance on unseen data.

**K-Fold CV:**
1. Split data into K equal folds
2. For each fold: train on K-1 folds, test on the remaining fold
3. Average the K test scores

$$CV_K = \frac{1}{K} \sum_{k=1}^{K} \text{Error}_k$$

**Leave-One-Out CV (LOOCV):** K = n (very expensive but unbiased)

**Stratified K-Fold:** Preserves class proportions in each fold (important for imbalanced data).

| Method | Bias | Variance | Cost |
|--------|------|----------|------|
| Train/Test Split | High | High | Low |
| 5-Fold CV | Medium | Medium | Medium |
| 10-Fold CV | Low | Low | High |
| LOOCV | Very Low | High | Very High |

> **Key Insight:** Use 5 or 10-fold CV for model evaluation and hyperparameter tuning. Never tune hyperparameters on the test set — use a validation set or nested CV.


In [ ]:
# K-Fold Cross-Validation demonstration
from sklearn.ensemble import RandomForestClassifier

knn = KNeighborsClassifier(n_neighbors=best_k)
rf  = RandomForestClassifier(n_estimators=200, random_state=42)

results = []
for name, model in [('KNN', knn), ('Random Forest', rf)]:
    cv_scores = cross_val_score(model, X_scaled, y, cv=10, scoring='accuracy')
    results.append({'Model': name, 'CV Accuracy': cv_scores.mean(),
                    'Std Dev': cv_scores.std(), 'Min': cv_scores.min(), 'Max': cv_scores.max()})

print("10-Fold Cross-Validation Results:")
print(pd.DataFrame(results).to_string(index=False))


---

## 3. The Bias-Variance Trade-off

### 📚 Theory

**Total prediction error decomposes into:**
$$E[(y - \hat{f}(x))^2] = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

| Component | Description | High when |
|-----------|-------------|-----------|
| **Bias** | Model is systematically wrong | Model is too simple (underfitting) |
| **Variance** | Model fluctuates with training data | Model is too complex (overfitting) |
| **Irreducible Error** | Noise in the data | Always present |

**Trade-off:**
- More complex model → lower bias, higher variance
- Simpler model → higher bias, lower variance

**Solution:** Choose complexity that minimises **total error** on validation data.

> **Key Insight:** The bias-variance trade-off is the fundamental tension in ML. Cross-validation and regularisation are the main tools for navigating it.


In [ ]:
# Demonstrate bias-variance trade-off with polynomial regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(42)
n = 50
X_bv = np.linspace(0, 1, n).reshape(-1, 1)
y_bv = np.sin(2 * np.pi * X_bv.ravel()) + np.random.randn(n) * 0.3

train_errors, val_errors = [], []
degrees = range(1, 12)

for deg in degrees:
    pipe = make_pipeline(PolynomialFeatures(deg), Ridge(alpha=0.0001))
    tr_scores = cross_val_score(pipe, X_bv, y_bv, cv=5, scoring='neg_mean_squared_error')
    train_err = -cross_val_score(pipe, X_bv, y_bv, cv=5,
                                  scoring='neg_mean_squared_error', return_train_score=False)
    train_errors.append(-tr_scores.mean())
    val_errors.append(train_err.mean())

plt.figure(figsize=(8, 4))
plt.plot(degrees, train_errors, 'b-o', label='CV Training Error')
plt.xlabel('Polynomial Degree')
plt.ylabel('Mean Squared Error')
plt.title('Bias-Variance Trade-off: Polynomial Degree vs Error')
plt.legend()
plt.tight_layout()
plt.show()


---

## 4. Regularised Regression

### 📚 Theory

**Regularisation** adds a penalty term to the loss function to prevent overfitting:

| Method | Loss Function | Effect |
|--------|-------------|--------|
| **OLS** | $\sum(y-\hat{y})^2$ | No regularisation |
| **Ridge (L2)** | $\sum(y-\hat{y})^2 + \lambda\sum\beta_j^2$ | Shrinks all coefficients; never zero |
| **Lasso (L1)** | $\sum(y-\hat{y})^2 + \lambda\sum|\beta_j|$ | Sets some coefficients to **zero** (feature selection) |
| **Elastic Net** | Ridge + Lasso combined | Best of both worlds |

**Hyperparameter λ (alpha):**  
- λ = 0 → OLS (no regularisation)
- λ → ∞ → all coefficients → 0

> **Key Insight:** Use **Lasso** when you expect many features to be irrelevant (sparse solution). Use **Ridge** when you expect most features to contribute. Use **Elastic Net** when you're unsure.


In [ ]:
# Regularised regression on house price data
house = pd.read_csv(HOUSE_CSV, sep='	')
features = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade',
            'NbrLivUnits', 'SqFtFinBasement', 'YrBuilt', 'YrRenovated']
X_house = house[features].fillna(0)
y_house = house['AdjSalePrice']

X_train, X_test, y_train, y_test = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

scaler_h = StandardScaler()
X_train_s = scaler_h.fit_transform(X_train)
X_test_s  = scaler_h.transform(X_test)

models = {
    'Ridge (λ=1)': Ridge(alpha=1.0),
    'Ridge (λ=100)': Ridge(alpha=100.0),
    'Lasso (λ=100)': Lasso(alpha=100.0),
    'Elastic Net': ElasticNet(alpha=100.0, l1_ratio=0.5),
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train_s)))
    test_rmse  = np.sqrt(mean_squared_error(y_test, model.predict(X_test_s)))
    n_zero = sum(abs(c) < 1e-6 for c in model.coef_)
    results.append({'Model': name, 'Train RMSE': train_rmse, 'Test RMSE': test_rmse,
                    'Zero Coefs': n_zero})

print(pd.DataFrame(results).to_string(index=False))


In [ ]:
# Lasso regularisation path (coefficient shrinkage)
alphas = np.logspace(1, 5, 50)
coef_paths = []
for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_s, y_train)
    coef_paths.append(lasso.coef_)

coef_paths = np.array(coef_paths)

plt.figure(figsize=(10, 5))
for i, feat in enumerate(features):
    plt.plot(np.log10(alphas), coef_paths[:, i], label=feat)
plt.xlabel('log10(λ)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Regularisation Path')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Cross-validated Lasso (LassoCV)
lasso_cv = LassoCV(cv=5, max_iter=5000)
lasso_cv.fit(X_train_s, y_train)
print(f"Best Lasso α (lambda): {lasso_cv.alpha_:.2f}")
print(f"Non-zero coefficients: {sum(abs(lasso_cv.coef_) > 1e-6)} / {len(features)}")
print("
Selected features:")
for feat, coef in zip(features, lasso_cv.coef_):
    if abs(coef) > 1e-6:
        print(f"  {feat}: {coef:,.2f}")


---

## 5. Model Interpretability

### 📚 Theory

As models become more complex, interpretability decreases. Tools to interpret black-box models:

- **Partial Dependence Plots (PDP):** Marginal effect of one feature on prediction
- **SHAP Values:** Shapley values explaining each prediction's feature contributions
- **Permutation Importance:** Drop in performance when feature is randomly shuffled

**SHAP:**  
$$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} [f(S \cup \{j\}) - f(S)]$$

> **Key Insight:** Always interpret model decisions, especially in high-stakes domains (credit, healthcare). SHAP is the gold standard for local (per-prediction) and global (model-level) explanation.


In [ ]:
# Partial Dependence: SqFtTotLiving effect on house price
from sklearn.inspection import PartialDependenceDisplay
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
print(f"Gradient Boosting Test RMSE: {np.sqrt(mean_squared_error(y_test, gb.predict(X_test))):,.0f}")


In [ ]:
# Partial Dependence Plot for top 2 features
fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    gb, X_train, features=[0, 4],
    feature_names=features, ax=ax
)
plt.suptitle('Partial Dependence Plots')
plt.tight_layout()
plt.show()


---

## ✅ Chapter 6 Key Takeaways

1. **KNN:** Non-parametric and intuitive; always scale features; struggles in high dimensions.
2. **Cross-Validation:** Use 5 or 10-fold CV for honest performance estimates; never tune on the test set.
3. **Bias-Variance Trade-off:** The key tension in ML — regularisation and cross-validation navigate it.
4. **Ridge:** Shrinks all coefficients; good when all features matter.
5. **Lasso:** Performs feature selection by zeroing out coefficients; good for sparse problems.
6. **Elastic Net:** Combines Ridge and Lasso; robust to correlated features.
7. **Interpretability:** Use SHAP, PDP, or permutation importance to understand model decisions.

---
*Reference: Chapter 6, Practical Statistics for Data Scientists (O'Reilly)*
